In [16]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# **Milestone - 5**

# **Setup**

In [17]:
!pip install transformers -q

import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
LABEL_MAP   = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

# Load DeBERTa
deberta_tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-small')
deberta_model     = AutoModelForSequenceClassification.from_pretrained(
    'microsoft/deberta-v3-small', num_labels=5
)
deberta_model.eval()

# Load RoBERTa
roberta_tokenizer = AutoTokenizer.from_pretrained('roberta-base')
roberta_model     = AutoModelForSequenceClassification.from_pretrained(
    'roberta-base', num_labels=5
)
roberta_model.eval()

print("Models loaded.")

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Models loaded.


**Inference helper functions**

In [19]:
def get_probs_deberta(prompt, options):
    text = prompt + " " + " ".join([f"{opt}: {val}" for opt, val in zip(OPTION_COLS, options)])
    inputs = deberta_tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    with torch.no_grad():
        logits = deberta_model(**inputs).logits
    return F.softmax(logits, dim=-1)[0].numpy()

def get_probs_roberta(prompt, options):
    text = prompt + " " + " ".join([f"{opt}: {val}" for opt, val in zip(OPTION_COLS, options)])
    inputs = roberta_tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    with torch.no_grad():
        logits = roberta_model(**inputs).logits
    return F.softmax(logits, dim=-1)[0].numpy()

def get_options(row):
    return [str(row[c]) for c in OPTION_COLS]

print("Helper functions ready.")

Helper functions ready.


Q 1) **DeBERTa highest probability option for row 25**

In [20]:
row25   = train.iloc[25]
prompt  = str(row25['prompt'])
options = get_options(row25)

probs_deberta = get_probs_deberta(prompt, options)

top_idx      = np.argmax(probs_deberta)
top_option   = LABEL_MAP[top_idx]
top_prob     = probs_deberta[top_idx]

print("DeBERTa probabilities:")
for i, (opt, prob) in enumerate(zip(OPTION_COLS, probs_deberta)):
    print(f"  {opt}: {prob:.4f}")

print(f"\nAnswer Q1 — Option: {top_option}, Probability: {round(top_prob, 4)}")

DeBERTa probabilities:
  A: 0.1755
  B: 0.2571
  C: 0.1689
  D: 0.1414
  E: 0.2571

Answer Q1 — Option: B, Probability: 0.257080078125


Q 2) **Simple average ensemble, highest option**  

In [21]:
probs_roberta = get_probs_roberta(prompt, options)

avg_probs = (probs_deberta + probs_roberta) / 2

top_idx_avg  = np.argmax(avg_probs)
top_opt_avg  = LABEL_MAP[top_idx_avg]

print("Averaged probabilities:")
for opt, prob in zip(OPTION_COLS, avg_probs):
    print(f"  {opt}: {prob:.4f}")

print(f"\nAnswer Q2 — Highest option after averaging: {top_opt_avg}")

Averaged probabilities:
  A: 0.1884
  B: 0.2301
  C: 0.1992
  D: 0.1616
  E: 0.2206

Answer Q2 — Highest option after averaging: B


Q 3) **Weighted ensemble (0.7 DeBERTa + 0.3 RoBERTa)**

In [22]:
weighted_probs = 0.7 * probs_deberta + 0.3 * probs_roberta

top_idx_w  = np.argmax(weighted_probs)
top_opt_w  = LABEL_MAP[top_idx_w]

print("Weighted probabilities:")
for opt, prob in zip(OPTION_COLS, weighted_probs):
    print(f"  {opt}: {prob:.4f}")

print(f"\nAnswer Q3 — Ranked first after weighted ensemble: {top_opt_w}")

Weighted probabilities:
  A: 0.1833
  B: 0.2410
  C: 0.1871
  D: 0.1536
  E: 0.2353

Answer Q3 — Ranked first after weighted ensemble: B


Q 4) **Top-3 prediction string for row 25**

In [24]:
top3_idx  = np.argsort(weighted_probs)[::-1][:3]
top3_opts = [LABEL_MAP[i] for i in top3_idx]
top3_str  = ' '.join(top3_opts)

print(f"Answer Q4 — Top-3 prediction: {top3_str}")

Answer Q4 — Top-3 prediction: B E C


Q 5) **Total prediction rows in submission.csv**

In [25]:
from tqdm import tqdm

test_preds = []

for _, row in tqdm(test.iterrows(), total=len(test)):
    prompt  = str(row['prompt'])
    options = get_options(row)

    p_deb = get_probs_deberta(prompt, options)
    p_rob = get_probs_roberta(prompt, options)
    w_p   = 0.7 * p_deb + 0.3 * p_rob

    top3  = [LABEL_MAP[i] for i in np.argsort(w_p)[::-1][:3]]
    test_preds.append(' '.join(top3))

submission = pd.DataFrame({
    'id'         : test['id'],
    'prediction' : test_preds
})
submission.to_csv('submission.csv', index=False)

print(f"Answer Q5 — Total prediction rows: {len(submission)}")
print(submission.head())

100%|██████████| 500/500 [13:54<00:00,  1.67s/it]

Answer Q5 — Total prediction rows: 500
   id prediction
0   1      E C B
1   2      C B E
2   3      E C B
3   4      B C A
4   5      C B E


Q 6) **TTA: rows with different Top-1 after augmentation (first 50)**

In [34]:
tta_diff = 0

for i in range(50):
    row     = test.iloc[i]
    prompt  = str(row['prompt'])
    options = get_options(row)

    # Original
    p_orig = get_probs_deberta(prompt, options)

    # Augmented
    aug_prompt = "Answer the following multiple-choice question carefully: " + prompt
    p_aug      = get_probs_deberta(aug_prompt, options)

    # Average
    p_avg = (p_orig + p_aug) / 2

    top1_orig = LABEL_MAP[np.argmax(p_orig)]
    top1_tta  = LABEL_MAP[np.argmax(p_avg)]

    if top1_orig != top1_tta:
        tta_diff += 1

print(f"Answer Q6 — Rows with different Top-1 after TTA: {tta_diff}")

Answer Q6 — Rows with different Top-1 after TTA: 8


Q 7) **Rows with different Top-1: DeBERTa vs Weighted Ensemble (first 100)**

In [29]:
diff_count = 0

for i in range(100):
    row     = test.iloc[i]
    prompt  = str(row['prompt'])
    options = get_options(row)

    p_deb = get_probs_deberta(prompt, options)
    p_rob = get_probs_roberta(prompt, options)
    w_p   = 0.7 * p_deb + 0.3 * p_rob

    top1_deb = LABEL_MAP[np.argmax(p_deb)]
    top1_ens = LABEL_MAP[np.argmax(w_p)]

    if top1_deb != top1_ens:
        diff_count += 1

print(f"Answer Q7 — Rows with different Top-1: {diff_count}")

Answer Q7 — Rows with different Top-1: 7


Q 8) **Rows with positive confidence gain (first 100)**

In [31]:
pos_gain = 0

for i in range(100):
    row     = test.iloc[i]
    prompt  = str(row['prompt'])
    options = get_options(row)

    p_deb = get_probs_deberta(prompt, options)
    p_rob = get_probs_roberta(prompt, options)
    w_p   = 0.7 * p_deb + 0.3 * p_rob

    conf_deb = np.max(p_deb)
    conf_ens = np.max(w_p)

    if conf_ens - conf_deb > 0:
        pos_gain += 1

print(f"Answer Q8 — Rows with positive confidence gain: {pos_gain}")

Answer Q8 — Rows with positive confidence gain: 2


Q9)  **Rows with at least one change in Top-3 (first 100)**

In [32]:
top3_diff = 0

for i in range(100):
    row     = test.iloc[i]
    prompt  = str(row['prompt'])
    options = get_options(row)

    p_deb = get_probs_deberta(prompt, options)
    p_rob = get_probs_roberta(prompt, options)
    w_p   = 0.7 * p_deb + 0.3 * p_rob

    top3_deb = [LABEL_MAP[j] for j in np.argsort(p_deb)[::-1][:3]]
    top3_ens = [LABEL_MAP[j] for j in np.argsort(w_p)[::-1][:3]]

    if top3_deb != top3_ens:
        top3_diff += 1

print(f"Answer Q9 — Rows with changed Top-3: {top3_diff}")

Answer Q9 — Rows with changed Top-3: 26


Q10) **MAP@3 of weighted ensemble on first 100 train rows**

In [35]:
def map_at_3(ground_truth, predictions):
    score = 0.0
    for k, pred in enumerate(predictions[:3], start=1):
        if pred == ground_truth:
            score = 1.0 / k
            break
    return score

scores = []

for i in range(100):
    row     = train.iloc[i]
    prompt  = str(row['prompt'])
    options = get_options(row)
    answer  = row['answer']

    p_deb = get_probs_deberta(prompt, options)
    p_rob = get_probs_roberta(prompt, options)
    w_p   = 0.7 * p_deb + 0.3 * p_rob

    top3 = [LABEL_MAP[j] for j in np.argsort(w_p)[::-1][:3]]
    scores.append(map_at_3(answer, top3))

final_map3 = np.mean(scores)
print(f"Answer Q10 — MAP@3: {round(final_map3, 4)}")

Answer Q10 — MAP@3: 0.425
